# Suite PY — The Python language

Python 3 as a tree language (`mbse.Programs.Python.Syntax`), whose kinds follow Python's own `ast` module: its kinds
and categories, the identifiers it accepts, each kind's own checks, and where each kind and feature exists, which the
versions check.

In [ ]:
from mbse.Programs.Python import Python312, Python314, Syntax as S
from mbse.Programs.Python._Standard import PythonStandard

L = S.LANGUAGE


def version(minor):
    return PythonStandard(3, minor)


def n(spelling):
    return S.Identifier(spelling=spelling)


def v(spelling):
    return S.Name(id=n(spelling))


def c(spelling):
    return S.Constant(spelling=spelling)

## PY-01 · Every kind belongs to the language, the grammar lists them, and their properties are in source order

In [ ]:
kinds = L.kinds()
assert list(kinds) == [k.__name__ for k in S.KINDS] and len(kinds) == 86
assert list(L.categories()) == ["Statement", "Expression", "Pattern", "TypeParameter"]
assert all(k.NAME == f"Programs.Python.{k.KIND}" for k in S.KINDS)
grammar = L.grammar()
assert grammar["language"] == "Python" and grammar["base"] == {"Python": 300}
assert next(k for k in grammar["kinds"] if k["kind"] == "TemplateStr")["since"] == {"Python": 314}
assert next(k for k in grammar["kinds"] if k["kind"] == "Module")["category"] == "SyntaxNode"
assert S.py(12) == {"Python": 312} and isinstance(S.Comment(), S.Statement)
assert [f.name for f in S.FunctionDef.PROPERTIES] == ["decorator_list", "name", "type_params", "args", "returns", "body"]
assert [f.name for f in S.Arg.PROPERTIES] == ["arg", "annotation", "default_value"]
print(len(kinds), "kinds")

## PY-02 · Identifiers are Unicode identifiers other than keywords; soft keywords are identifiers

In [ ]:
for spelling in ("x_1", "Größe", "_", "match", "case", "type", "lazy", "print"):
    assert L.validate(n(spelling)) == [], spelling
assert L.validate(n("1x")) == ["'1x' is not an identifier"]
assert L.validate(n("a-b")) == ["'a-b' is not an identifier"]
assert L.validate(n("")) == ["'' is not an identifier"]
assert L.validate(n("class")) == ["'class' is a keyword"]
assert L.validate(n("None")) == ["'None' is a keyword"]
assert L.validate(n(3)) == ["Identifier.spelling must be a str, got int"]
assert L.validate(S.Identifier()) == ["an Identifier needs a spelling"]
print("ok")

## PY-03 · Each kind's own checks: what its lists must hold, and what goes together

In [ ]:
gen = [S.Comprehension(target=v("x"), iter=v("y"))]


def body():
    return [S.Pass()]


def comments():
    return [S.Comment(text=" c")]


problems = {
    S.DottedName(): "a DottedName needs a name",
    S.ConcatenatedString(values=[c("'a'")]): "a ConcatenatedString needs two strings",
    S.Set(): "a Set needs an element",
    S.ListComp(elt=v("x")): "a ListComp needs a for clause",
    S.SetComp(elt=v("x")): "a SetComp needs a for clause",
    S.GeneratorExp(elt=v("x")): "a GeneratorExp needs a for clause",
    S.DictComp(key=v("x")): "a DictComp needs a for clause",
    S.Compare(left=v("a")): "a Compare needs a comparison",
    S.BoolOp(op="and", values=[v("a")]): "a BoolOp needs two values",
    S.Lambda(args=S.Arguments(args=[S.Arg(arg=n("a"), annotation=v("int"))]), body=v("a")):
        "a Lambda's parameters have no annotations",
    S.Assign(value=v("a")): "an Assign needs a target",
    S.Delete(): "a Delete needs a target",
    S.Import(): "an Import needs a name",
    S.ImportFrom(module=S.DottedName(names=[n("m")])): "an ImportFrom needs a name",
    S.ImportFrom(names=[S.Alias()]): "an ImportFrom needs a module or a level",
    S.Global(): "a Global needs a name",
    S.Raise(cause=v("c")): "a Raise with a cause needs an exc",
    S.Nonlocal(): "a Nonlocal needs a name",
    S.If(test=v("a"), body=comments()): "an If needs a statement in its body",
    S.If(test=v("a"), body=body(), orelse=comments()): "an If needs a statement in its orelse",
    S.While(test=v("a")): "a While needs a statement in its body",
    S.For(target=v("a"), iter=v("b")): "a For needs a statement in its body",
    S.AsyncFor(target=v("a"), iter=v("b")): "an AsyncFor needs a statement in its body",
    S.ExceptHandler(): "an ExceptHandler needs a statement in its body",
    S.Try(body=body()): "a Try needs a handler or a finalbody",
    S.Try(body=body(), orelse=body(), finalbody=body()): "a Try with an orelse needs a handler",
    S.TryStar(body=body()): "a TryStar needs a handler or a finalbody",
    S.With(body=body()): "a With needs an item",
    S.AsyncWith(items=[S.WithItem(context_expr=v("a"))]): "an AsyncWith needs a statement in its body",
    S.MatchCase(pattern=S.MatchAs()): "a MatchCase needs a statement in its body",
    S.Match(subject=v("a")): "a Match needs a case",
    S.FunctionDef(name=n("f"), args=S.Arguments()): "a FunctionDef needs a statement in its body",
    S.AsyncFunctionDef(name=n("f"), args=S.Arguments()): "an AsyncFunctionDef needs a statement in its body",
    S.ClassDef(name=n("C")): "a ClassDef needs a statement in its body",
    S.MatchMapping(keys=[c("1")]): "a MatchMapping needs a pattern for each key",
    S.MatchClass(cls=v("C"), kwd_attrs=[n("a")]): "a MatchClass needs a pattern for each keyword",
    S.MatchOr(patterns=[S.MatchAs()]): "a MatchOr needs two patterns",
    S.JoinedStr(prefix="fb", quote="'"): "JoinedStr.prefix cannot be 'fb'",
    S.TemplateStr(prefix="tf", quote="'"): "TemplateStr.prefix cannot be 'tf'",
}
for node, problem in problems.items():
    assert L.validate(node) == [problem], (node, L.validate(node))
assert L.validate(S.JoinedStr(prefix=1, quote="'")) == ["JoinedStr.prefix must be a str, got int"]
assert L.validate(S.Lambda(args=None, body=v("a"))) == ["a Lambda needs an args"]
for prefix in ("f", "F", "rf", "fR", "Rf"):
    assert L.validate(S.JoinedStr(prefix=prefix, quote='"')) == []
assert L.validate(S.TemplateStr(prefix="Tr", quote='"""')) == []
fine = [
    S.Try(body=body(), handlers=[S.ExceptHandler(body=body())], orelse=body()),
    S.Try(body=body(), finalbody=comments() + body()),
    S.ImportFrom(level=1, names=[S.Alias()]), S.BoolOp(op="or", values=[v("a"), v("b")]),
    S.Lambda(args=S.Arguments(vararg=S.Arg(arg=n("a"))), body=v("a")),
]
for node in fine:
    assert L.validate(node) == [], (node, L.validate(node))
print(len(problems), "problems")

## PY-04 · Where kinds and features exist: since which version of Python

In [ ]:
def needs(node, minor, problem):
    """`node` is fine in Python 3.`minor` and has `problem` in the version before."""
    assert version(minor).check(node) == [], (node, version(minor).check(node))
    assert version(minor - 1).check(node) == [problem], version(minor - 1).check(node)


assert Python314.STANDARD.name() == "Python 3.14" and repr(Python312.STANDARD) == "<standard Python 3.12>"
assert Python314.STANDARD.version == 314 and version(0).name() == "Python 3.0"
generator = S.Comprehension(target=v("x"), iter=v("y"))
needs(S.Await(value=v("a")), 5, "Await needs Python 3.5")
needs(S.BinOp(left=v("a"), op="@", right=v("b")), 5, "BinOp.op '@' needs Python 3.5")
needs(S.AugAssign(target=v("a"), op="@", value=v("b")), 5, "AugAssign.op '@' needs Python 3.5")
needs(S.AnnAssign(target=v("a"), annotation=v("int")), 6, "AnnAssign needs Python 3.6")
needs(S.JoinedStr(prefix="f", quote="'"), 6, "JoinedStr needs Python 3.6")
needs(S.Comprehension(is_async=True, target=v("x"), iter=v("y")), 6, "Comprehension.is_async needs Python 3.6")
needs(S.NamedExpr(target=v("a"), value=v("b")), 8, "NamedExpr needs Python 3.8")
needs(S.Arguments(posonlyargs=[S.Arg(arg=n("a"))]), 8, "Arguments.posonlyargs needs Python 3.8")
needs(S.FormattedValue(value=v("a"), debug=True), 8, "FormattedValue.debug needs Python 3.8")
decorated = S.FunctionDef(decorator_list=[S.Subscript(value=v("d"), slice=c("0"))], name=n("f"), args=S.Arguments(),
                          body=body())
needs(decorated, 9, "FunctionDef with a decorator that is not a dotted name needs Python 3.9")
dotted = S.Call(func=S.Attribute(value=v("d"), attr=n("e")))
assert version(8).check(S.ClassDef(decorator_list=[v("d"), dotted], name=n("C"), body=body())) == []
needs(S.MatchSingleton(value="None"), 10, "MatchSingleton needs Python 3.10")
needs(S.Match(subject=v("a")), 10, "Match needs Python 3.10")
needs(S.TryStar(), 11, "TryStar needs Python 3.11")
needs(S.Subscript(value=v("a"), slice=S.Starred(value=v("b"))), 11,
      "Subscript with an unpacked index needs Python 3.11")
needs(S.Subscript(value=v("a"), slice=S.Tuple(elts=[v("i"), S.Starred(value=v("b"))])), 11,
      "Subscript with an unpacked index needs Python 3.11")
assert version(10).check(S.Subscript(value=v("a"), slice=S.Tuple(elts=[v("i")]))) == []
needs(S.TypeVar(name=n("T")), 12, "TypeVar needs Python 3.12")
generic = S.ClassDef(name=n("C"), type_params=[S.TypeVar(name=n("T"))], body=body())
assert version(11).check(generic) == ["ClassDef.type_params needs Python 3.12",
                                     "type_params[0]: TypeVar needs Python 3.12"]
needs(S.ParamSpec(name=n("P"), default_value=v("a")), 13, "ParamSpec.default_value needs Python 3.13")
needs(S.TypeVarTuple(name=n("Ts"), default_value=v("a")), 13, "TypeVarTuple.default_value needs Python 3.13")
needs(S.TemplateStr(prefix="t", quote="'"), 14, "TemplateStr needs Python 3.14")
needs(S.ExceptHandler(type=S.Tuple(elts=[v("A"), v("B")]), body=body()), 14,
      "ExceptHandler with unparenthesized types needs Python 3.14")
assert version(13).check(S.ExceptHandler(type=S.Parenthesized(value=S.Tuple(elts=[v("A")])), body=body())) == []
needs(S.Import(is_lazy=True, names=[S.Alias(name=S.DottedName(names=[n("m")]))]), 15,
      "Import.is_lazy needs Python 3.15")
needs(S.ImportFrom(is_lazy=True, level=1, names=[S.Alias()]), 15, "ImportFrom.is_lazy needs Python 3.15")
needs(S.ListComp(elt=S.Starred(value=v("x")), generators=[generator]), 15,
      "ListComp of unpacked elements needs Python 3.15")
needs(S.GeneratorExp(elt=S.Starred(value=v("x")), generators=[generator]), 15,
      "GeneratorExp of unpacked elements needs Python 3.15")
needs(S.SetComp(elt=S.Starred(value=v("x")), generators=[generator]), 15,
      "SetComp of unpacked elements needs Python 3.15")
needs(S.DictComp(key=v("x"), generators=[generator]), 15, "DictComp of unpacked mappings needs Python 3.15")
assert version(14).check(S.DictComp(key=v("x"), value=v("y"), generators=[generator])) == []
assert Python312.check(S.Module(body=[S.Expr(value=S.TemplateStr(prefix="t", quote="'"))])) == [
    "body[0].value: TemplateStr needs Python 3.14"]
print("ok")